# 01 Downloading the photos

Inside Airbnb only gives a link to each listing's main photo, not the photo itself. So I download all of them once here and save them, and the later notebooks just read the saved copies. That way the data doesn't change if a link breaks halfway through the project.

On Colab everything gets saved to Google Drive, so if the session disconnects I don't lose anything. If you run this anywhere else it saves to a local folder instead, and it downloads everything from the public links, so it can be rerun from scratch.

The photos are only used as model input. They aren't shared or put on GitHub.

In [ ]:
from pathlib import Path

# On Colab I save to Drive so nothing is lost if the session drops.
# Anywhere else it just uses a local folder.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/airbnb_p22")
except ImportError:
    PROJECT_DIR = Path("airbnb_p22")

RAW_DIR = PROJECT_DIR / "data" / "raw"
PHOTO_DIR = PROJECT_DIR / "data" / "photos_480"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PHOTO_DIR.mkdir(parents=True, exist_ok=True)
print("Saving to:", PROJECT_DIR.resolve())

Mounted at /content/drive
Saving to: /content/drive/MyDrive/airbnb_p22


## Getting the listings file

I'm using the Melbourne snapshot from 16 September 2026, same as in our proposal. I keep a copy because Inside Airbnb removes older snapshots after a while, and I want every notebook to use exactly the same file. This file still has the host columns in it, so it stays on Drive and doesn't go on GitHub.

In [ ]:
import shutil
import tempfile
import urllib.request

import pandas as pd

URL = ("https://data.insideairbnb.com/australia/vic/melbourne/2026-09-16/"
       "data/listings.csv.gz")
RAW_FILE = RAW_DIR / "listings_2026-09-16.csv.gz"

# only download if we don't already have it
if not RAW_FILE.exists():
    tmp = Path(tempfile.gettempdir()) / "listings.csv.gz"
    try:
        urllib.request.urlretrieve(URL, tmp)
        shutil.copy(tmp, RAW_FILE)
    except Exception as e:
        raise RuntimeError(f"Download failed: {e}")

try:
    raw = pd.read_csv(RAW_FILE, compression="gzip", low_memory=False)
except (OSError, pd.errors.ParserError) as e:
    raise RuntimeError(f"Couldn't read {RAW_FILE}: {e}")
print(f"{len(raw):,} listings, {raw.shape[1]} columns")

# we can only label listings that have a price, so only those photos matter
price = pd.to_numeric(raw["price"].astype(str).str.replace(r"[$,]", "", regex=True),
                      errors="coerce")
photos = raw.loc[(price > 0) & raw["picture_url"].notna(), ["id", "picture_url"]]
# sorting by id means shard 0 always has the same listings, even after a restart
photos = photos.sort_values("id").reset_index(drop=True)
print(f"{len(photos):,} listings have a price and a photo link")

25,715 listings, 90 columns
19,004 listings have a price and a photo link


## Download functions

My first run asked the server for 480 pixel wide copies, but it ignored that and sent full-size photos (median 3,000 pixels wide, 55 GB in total). So now each photo gets shrunk straight after download so its longest side is 480 pixels, keeping its shape. The model only takes 224 x 224, so 480 is still plenty. Phone photos that are stored sideways get rotated first, and every photo is saved once as a JPEG at quality 90.

Photos are saved in zip files of 1,000 (shards). If Colab disconnects, I just run everything again and it skips the shards that are already done. Each zip also has a manifest.csv saying which photos worked and why the others failed.

In [ ]:
import io
import math
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import requests
from PIL import Image, ImageOps

SHARD_SIZE = 1000
MAX_SIDE = 480    # model input is only 224 x 224 so this is enough
N_WORKERS = 8     # didn't want to go higher in case the server starts blocking us
RETRIES = 2

# these are real listing photos, a few are just very big (100+ megapixels),
# so I raise Pillow's limit instead of getting warnings or errors
Image.MAX_IMAGE_PIXELS = 200_000_000


def shrink(data):
    img = Image.open(io.BytesIO(data))
    img.draft("RGB", (MAX_SIDE, MAX_SIDE))   # JPEG trick: decode at 1/2, 1/4 or 1/8 size, much faster
    img = ImageOps.exif_transpose(img).convert("RGB")   # phone photos can be stored sideways
    img.thumbnail((MAX_SIDE, MAX_SIDE))   # longest side 480, keeps the shape
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return buf.getvalue(), img.width, img.height


def fetch_photo(listing_id, url):
    error = ""
    for attempt in range(RETRIES + 1):
        try:
            r = requests.get(url, timeout=30)   # longer timeout, the files are full size
            r.raise_for_status()
            data, w, h = shrink(r.content)
            return listing_id, data, w, h, ""
        except Exception as e:
            error = type(e).__name__
            time.sleep(1 + attempt)
    return listing_id, None, 0, 0, error


def shard_file(k):
    return PHOTO_DIR / f"photos_{k:03d}.zip"


def shard_done(path):
    # manifest.csv is written last, so if it's missing the shard got cut off
    if not path.exists():
        return False
    try:
        with zipfile.ZipFile(path) as z:
            return "manifest.csv" in z.namelist()
    except (zipfile.BadZipFile, OSError):
        return False


def download_shard(k, rows):
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        results = list(pool.map(lambda row: fetch_photo(*row), rows))

    records = []
    # Drive is really slow with lots of small files, so I build the zip
    # locally first and then copy the one file across
    tmp = Path(tempfile.gettempdir()) / f"photos_{k:03d}.zip"
    try:
        with zipfile.ZipFile(tmp, "w") as z:
            for listing_id, data, w, h, err in results:
                if data is not None:
                    z.writestr(f"{listing_id}.jpg", data)
                records.append({"id": listing_id, "ok": data is not None,
                                "width": w, "height": h, "error": err})
            z.writestr("manifest.csv", pd.DataFrame(records).to_csv(index=False))
        shutil.copy(tmp, shard_file(k))
        tmp.unlink()
    except OSError as e:
        raise RuntimeError(f"Couldn't save shard {k}: {e}")
    return pd.DataFrame(records)

## Shrinking the full-size photos I already downloaded

My first run saved full-size photos in data/photos. Instead of downloading everything again, this shrinks those copies into data/photos_480 using the same shrink function. Photos that failed to download stay marked as failed. A fresh run of this notebook doesn't need this cell, because the download now shrinks photos straight away.

In [ ]:
FULL_DIR = PROJECT_DIR / "data" / "photos"
BATCH = 100   # only 100 full-size photos in memory at a time, they're about 3 MB each


def try_shrink(data):
    if data is None:
        return None, 0, 0, ""
    try:
        return shrink(data) + ("",)
    except Exception as e:
        return None, 0, 0, type(e).__name__


def convert_shard(path):
    k = int(path.stem.split("_")[1])
    if shard_done(shard_file(k)):
        return f"shard {k + 1}: already converted, skipping"
    tmp = Path(tempfile.gettempdir()) / f"small_{k:03d}.zip"
    records = []
    try:
        with zipfile.ZipFile(path) as zin, zipfile.ZipFile(tmp, "w") as zout:
            old = pd.read_csv(zin.open("manifest.csv"))
            for start in range(0, len(old), BATCH):
                chunk = old.iloc[start:start + BATCH]
                blobs = [zin.read(f"{i}.jpg") if ok else None
                         for i, ok in zip(chunk["id"], chunk["ok"])]
                with ThreadPoolExecutor(max_workers=8) as pool:
                    results = list(pool.map(try_shrink, blobs))
                for (_, row), (data, w, h, err) in zip(chunk.iterrows(), results):
                    if data is not None:
                        zout.writestr(f"{row['id']}.jpg", data)
                    # keep the original download error for photos that never came through
                    records.append({"id": row["id"], "ok": data is not None, "width": w,
                                    "height": h, "error": err or row["error"]})
            zout.writestr("manifest.csv", pd.DataFrame(records).to_csv(index=False))
        shutil.copy(tmp, shard_file(k))
        tmp.unlink()
    except (zipfile.BadZipFile, KeyError, OSError) as e:
        raise RuntimeError(f"Couldn't convert {path.name}: {e}")
    return f"shard {k + 1}: {sum(r['ok'] for r in records)} photos shrunk"


start = time.time()
for path in sorted(FULL_DIR.glob("photos_*.zip")):
    print(convert_shard(path))
print(f"done in {(time.time() - start) / 60:.0f} min")

shard 1: already converted, skipping
shard 2: already converted, skipping
shard 20: 4 photos shrunk
done in 0 min


## Downloading everything

This took about 80 minutes on my first run. If it disconnects, just run all the cells again.

In [ ]:
rows = list(photos.itertuples(index=False, name=None))
n_shards = math.ceil(len(rows) / SHARD_SIZE)
start = time.time()

for k in range(n_shards):
    if shard_done(shard_file(k)):
        print(f"shard {k + 1}/{n_shards} already done, skipping")
        continue
    t0 = time.time()
    m = download_shard(k, rows[k * SHARD_SIZE:(k + 1) * SHARD_SIZE])
    print(f"shard {k + 1}/{n_shards}: {m['ok'].mean():.1%} ok, "
          f"{len(m) / (time.time() - t0):.1f} photos/s")

print(f"done in {(time.time() - start) / 60:.0f} min")

shard 1/20 already done, skipping
shard 2/20 already done, skipping
shard 3/20 already done, skipping
shard 4/20 already done, skipping
shard 5/20 already done, skipping
shard 6/20: 99.3% ok, 6.3 photos/s
shard 7/20: 99.2% ok, 6.5 photos/s
shard 8/20: 99.4% ok, 6.9 photos/s
shard 9/20: 99.1% ok, 7.2 photos/s
shard 10/20: 99.7% ok, 6.2 photos/s
shard 11/20: 99.2% ok, 6.3 photos/s
shard 12/20: 99.2% ok, 6.6 photos/s
shard 13/20: 99.6% ok, 6.7 photos/s
shard 14/20: 99.1% ok, 6.5 photos/s
shard 15/20: 98.4% ok, 6.8 photos/s
shard 16/20: 98.6% ok, 6.4 photos/s
shard 17/20: 98.1% ok, 6.1 photos/s
shard 18/20: 98.2% ok, 6.8 photos/s
shard 19/20: 95.2% ok, 5.6 photos/s
shard 20/20 already done, skipping
done in 36 min


## Checking the photos

Here I check how many photos downloaded, why the rest failed, and what shapes they are. The shape matters for the next step: the usual EfficientNet preprocessing crops the photo to a square, which cuts the sides off wide photos.

I also save one combined list of which listings have a photo (photo_manifest.csv). It only has listing ids and photo sizes, so it can go on GitHub, and anyone rerunning this can check they got the same photos.

In [ ]:
import matplotlib.pyplot as plt

parts = []
for path in sorted(PHOTO_DIR.glob("photos_*.zip")):
    try:
        with zipfile.ZipFile(path) as z:
            parts.append(pd.read_csv(z.open("manifest.csv")))
    except (zipfile.BadZipFile, KeyError, OSError) as e:
        print(f"problem with {path.name}: {e}")
manifest = pd.concat(parts, ignore_index=True)

ok = manifest[manifest["ok"]]
print(f"Downloaded {len(ok):,} of {len(manifest):,} photos ({manifest['ok'].mean():.1%})")
print("Why the rest failed:", manifest.loc[~manifest["ok"], "error"].value_counts().to_dict())

# checking shapes now because a square crop will cut the sides off wide photos
ratio = ok["width"] / ok["height"]
print(f"{ok.groupby(['width', 'height']).ngroups} different photo sizes")
print(f"Width/height ratio: median {ratio.median():.2f}, "
      f"min {ratio.min():.2f}, max {ratio.max():.2f}")
print(f"Landscape {(ratio > 1).mean():.1%}, portrait {(ratio < 1).mean():.1%}")

# the next notebooks need to know which listings actually have a photo
try:
    manifest.to_csv(PROJECT_DIR / "data" / "photo_manifest.csv", index=False)
except OSError as e:
    print(f"Couldn't save photo_manifest.csv: {e}")

fig, axes = plt.subplots(2, 6, figsize=(16, 5))
for ax, listing_id in zip(axes.ravel(), ok["id"].sample(12, random_state=42)):
    k = manifest.index[manifest["id"] == listing_id][0] // SHARD_SIZE  # rows are in shard order
    with zipfile.ZipFile(shard_file(k)) as z:
        ax.imshow(Image.open(z.open(f"{listing_id}.jpg")))
    ax.set_title(listing_id, fontsize=8)
    ax.axis("off")
plt.suptitle("Random sample of the downloaded photos")
plt.tight_layout()
plt.show()

Downloaded 18,802 of 19,004 photos (98.9%)
Why the rest failed: {'HTTPError': 197, 'UnidentifiedImageError': 5}
484 different photo sizes
Width/height ratio: median 1.50, min 0.39, max 3.43
Landscape 82.6%, portrait 16.3%


[Photo grid removed before publishing: the photos belong to the hosts. Run the cell to see it.]


In [ ]:
# the server ignored the 480px request on my first run, so checking every photo is now shrunk
big = ok[ok["width"] > MAX_SIDE]
print(f"{len(big):,} photos wider than {MAX_SIDE}px ({len(big) / len(ok):.1%})")
print(ok["width"].describe().round(0))
total_mb = sum(p.stat().st_size for p in PHOTO_DIR.glob("photos_*.zip")) / 1e6
print(f"Total size on Drive: {total_mb:,.0f} MB")

0 photos wider than 480px (0.0%)
count    18802.0
mean       458.0
std         52.0
min         53.0
25%        480.0
50%        480.0
75%        480.0
max        480.0
Name: width, dtype: float64
Total size on Drive: 800 MB


## Making sure everything is saved

Colab uploads files to Drive in the background. On my first run I deleted the runtime before the uploads finished, and also ran out of Drive space, so 15 of the 20 photo zips never reached Drive and had to be downloaded again (shards 6 to 19, on 6 November). This cell waits until every file is actually saved.

In [ ]:
# Colab uploads to Drive in the background, so this waits until every file has
# actually been saved before the runtime is closed (my first run lost 15 zips)
from google.colab import drive
drive.flush_and_unmount()
print("Everything saved to Drive")

Everything saved to Drive
